# Individual Policy Training F

This notebook trains the individual forward-simulation policies. This is not the meta policy.

It trains three separate PPO policies with one shared implementation:

- `lr_calculation`: selects read/retrieve mode, decision boundary, and feature slots.
- `lr_heuristic`: selects decision boundary and feature slots; there is no read/retrieve distinction.
- `dt_traversal`: selects decision boundary only.

Each environment includes the same five context values plus the raw previous five probabilities of being correct and raw previous five decision times. Missing history at the start of an episode is filled with `-1`.

## Observation And Action Spaces

Common observation prefix for all strategies:

1. normalized decision noise
2. normalized memory retrieval threshold
3. normalized opportunity cost
4. whether explanation is currently shown
5. episode progress
6. previous five probabilities assigned to the correct label, padded with `-1`
7. previous five prediction times, scaled by `/60` and padded with `-1`

The feature-selecting strategies use per-episode randomized feature slots. The environment maps those slots back to actual feature indices before calling the cognitive strategy and stores both slot-level and actual-feature selections during evaluation.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "rl_agents":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
from rl_agents.individual_policy_training_f import (
    STRATEGIES,
    StrategyTrainingConfig,
    evaluate_strategy_policy,
    default_parameter_sweep_values,
    feature_selection_sweep_summary,
    load_all_strategy_policies,
    load_strategy_bundles,
    load_strategy_policy,
    plot_parameter_sweep,
    summarize_parameter_sweep,
    sweep_strategy_parameters,
    plot_strategy_evaluations,
    strategy_observation_action_summary,
    summarize_strategy_evaluations,
    train_all_strategy_policies,
    train_strategy_policy,
)



## Configure Training

In [ ]:
strategy_config = StrategyTrainingConfig(
    data_dir=str(ROOT / "datasets"),
    output_root=str(ROOT / "outputs" / "unified_strategy_policy"),
    run_name="unified_strategy_demo",
    total_timesteps=3e5,
    n_envs=4,
    instances_per_episode=40,
    max_features=6,
    explanation_shown_ratio=0.5,
    decision_boundary_bins=5,
    decision_boundary_min=0.6,
    decision_boundary_max=1.8,
    decision_noise_min=0.3,
    decision_noise_max=0.7,
    memory_recall_threshold_min=-5.0,
    memory_recall_threshold_max=2.0,
    opportunity_cost_min=0.0,
    opportunity_cost_max=0.02,
    memory_recall_noise=0.5,
    retrieval_candidate_count=3,
    simulation_sample_count=16,
    history_window=5,
    randomize_feature_order_per_episode=True,
    seed=123,
)

display(strategy_observation_action_summary(strategy_config))
strategy_config

## Check Available Bundles

In [ ]:
bundles = load_strategy_bundles(Path(strategy_config.data_dir), strategy_config)
pd.DataFrame([
    {"app_id": b.app_id, "model_name": b.model_name, "n_instances": len(b.instance_ids)}
    for b in bundles
])

## Load Or Train Selected Policies

Set `SELECTED_STRATEGIES` to the individual forward policies you want to load/train. Set `REUSE_SAVED_MODELS = True` to evaluate existing `final_model.zip` files without retraining. Set it to `False` when you intentionally want a fresh PPO training run.


In [ ]:
SELECTED_STRATEGIES = [
    "lr_calculation",
    "lr_heuristic",
    "dt_traversal",
]
# Examples:
# SELECTED_STRATEGIES = ["dt_traversal"]
# SELECTED_STRATEGIES = ["lr_calculation", "dt_traversal"]

invalid_strategies = sorted(set(SELECTED_STRATEGIES) - set(STRATEGIES))
if invalid_strategies:
    raise ValueError(f"Unknown strategy name(s): {invalid_strategies}; expected one of {STRATEGIES}")

REUSE_SAVED_MODELS = False
MODEL_FILENAME = "final_model.zip"

strategy_results = {}
for strategy_name in SELECTED_STRATEGIES:
    config = StrategyTrainingConfig(**{**asdict(strategy_config), "strategy_name": strategy_name})
    if REUSE_SAVED_MODELS:
        strategy_results[strategy_name] = load_strategy_policy(
            config,
            model_filename=MODEL_FILENAME,
            device="cpu",
        )
    else:
        strategy_results[strategy_name] = train_strategy_policy(config)

pd.DataFrame([
    {
        "strategy_name": name,
        "run_dir": str(run_dir),
        "model_path": str(run_dir / "models" / MODEL_FILENAME),
        "n_bundles": len(strategy_bundles),
    }
    for name, (_model, run_dir, strategy_bundles) in strategy_results.items()
])


## Optional Single-Strategy Iteration

Use this cell instead of the all-strategy cell when you want a faster iteration loop.

In [ ]:
# one_config = StrategyTrainingConfig(**{**asdict(strategy_config), "strategy_name": "lr_calculation"})
# one_model, one_run_dir, one_bundles = train_strategy_policy(one_config)
# one_eval_df = evaluate_strategy_policy(one_model, one_bundles, one_config, n_episodes=50)
# one_eval_df.head()

## Evaluate And Visualize Trained Policies

This section works whether the policies were just trained or loaded from disk. It saves per-step evaluation rows to each strategy's `metrics/evaluation.csv` and shared comparison plots to `<run_name>/strategy_plots`.


In [ ]:
evaluation_tables = {}

for strategy_name, (model, run_dir, strategy_bundles) in strategy_results.items():
    config = StrategyTrainingConfig(**{**asdict(strategy_config), "strategy_name": strategy_name})
    eval_df = evaluate_strategy_policy(
        model,
        strategy_bundles,
        config,
        n_episodes=100,
        deterministic=True,
        sample_parameters=True,
    )
    eval_path = run_dir / "metrics" / "evaluation.csv"
    eval_df.to_csv(eval_path, index=False)
    evaluation_tables[strategy_name] = eval_df

summary_df = summarize_strategy_evaluations(evaluation_tables)
display(summary_df)

combined_eval_df = pd.concat(evaluation_tables.values(), ignore_index=True)
plot_dir = Path(strategy_config.output_root) / (strategy_config.run_name or "latest") / "strategy_plots"
figures = plot_strategy_evaluations(evaluation_tables, output_dir=plot_dir, rolling_window=25)
print(f"Saved {len(figures)} plot(s) to {plot_dir}")

display(combined_eval_df.head())


## Parameter Sensitivity Sweeps

These sweeps reuse the loaded/trained policies and evaluate them under fixed values of one cognitive parameter at a time. Unspecified parameters are held at the midpoint of their configured range, so each curve isolates one parameter.


In [ ]:
SWEEP_EPISODES = 25
sweep_values = {
    "decision_noise": [0.3, 0.35, 0.4, 0.45, 0.5],
    "opportunity_cost": [0.0, 0.005, 0.01, 0.015, 0.02],
    "memory_recall_threshold": [-1.0, -0.25, 0.5, 1.25, 2.0],
}

sweep_df = sweep_strategy_parameters(
    strategy_results,
    strategy_config,
    sweep_values=sweep_values,
    n_episodes=SWEEP_EPISODES,
    deterministic=True,
)

sweep_dir = Path(strategy_config.output_root) / (strategy_config.run_name or "latest") / "parameter_sweeps"
sweep_dir.mkdir(parents=True, exist_ok=True)
sweep_df.to_csv(sweep_dir / "parameter_sweep_rows.csv", index=False)

sweep_summary_df = summarize_parameter_sweep(sweep_df)
sweep_feature_df = feature_selection_sweep_summary(sweep_df)
sweep_summary_df.to_csv(sweep_dir / "parameter_sweep_summary.csv", index=False)
sweep_feature_df.to_csv(sweep_dir / "parameter_sweep_feature_selection.csv", index=False)

display(sweep_summary_df.head(15))
print(f"Saved sweep data to {sweep_dir}")


In [ ]:
sweep_figures = plot_parameter_sweep(
    sweep_df,
    output_dir=sweep_dir / "plots",
    show=True,
)
print(f"Saved {len(sweep_figures)} sweep plot(s) to {sweep_dir / 'plots'}")
